In [1]:
from contextlib import contextmanager
import datetime as dt
import json
import zipfile

import pandas as pd

import src

In [2]:
CHANNELS = [
    "@spdde",
    "@csumedia",
    "@FDP",
    "@cdutv",
    "@DieGruenen",
    "@AfDFraktionimBundestag",
    "@AfDTV",
    "@DIELINKE",
]

# Channel Metadata

In [3]:
def load_channel_meta(channel):
    path = src.PATH / "data/raw/yt" / channel / "channel_metadata.json"

    with path.open() as f:
        d = json.load(f)

    info = {
        "channel_uploader_id": d["uploader_id"],
        "channel_follower_count": d["channel_follower_count"],
        "channel_collected": dt.datetime.fromtimestamp(d["epoch"]),
        "channel_description": d["description"],
        "channel": d["channel"],
        "channel_id": d["channel_id"],
        "channel_url": d["channel_url"],
    }

    return info


def create_channel_frame(channels):
    results = []
    for channel in channels:
        info = load_channel_meta(channel)
        results.append(info)

    df = pd.DataFrame(results)

    return df


channels = create_channel_frame(CHANNELS)
channels.head()

,channel_uploader_id,channel_follower_count,channel_collected,channel_description,channel,channel_id,channel_url
0,@spdde,34500,2025-02-24 07:03:31,Wir machen soziale Politik für Dich. Abonniere...,SPD,UCSmbK1WtpYn2sOGLvSSXkKw,https://www.youtube.com/channel/UCSmbK1WtpYn2s...
1,@csumedia,6610,2025-02-24 08:44:41,Hallo und herzlich Willkommen auf unserem YouT...,CSU,UC5AagLvRz7ejBrONZVaA13Q,https://www.youtube.com/channel/UC5AagLvRz7ejB...
2,@FDP,28900,2025-02-24 12:38:31,💛 Aus Liebe zur Freiheit\n\nIMPRESSUM \n\nVera...,FDP,UC-sMkrfoQDH-xzMxPNckGFw,https://www.youtube.com/channel/UC-sMkrfoQDH-x...
3,@cdutv,30000,2025-02-24 09:23:27,Die CDU ist die Volkspartei der Mitte. Seit 19...,CDU,UCKyWIEse3u7ExKfAWuDMVnw,https://www.youtube.com/channel/UCKyWIEse3u7Ex...
4,@DieGruenen,35400,2025-02-24 05:48:17,BÜNDNIS 90/DIE GRÜNEN \nPlatz vor dem Neuen To...,BÜNDNIS 90/DIE GRÜNEN,UC7TAA2WYlPfb6eDJCeX4u0w,https://www.youtube.com/channel/UC7TAA2WYlPfb6...


In [4]:
CHANNEL_FILE = src.PATH / "data/yt_metadata/channels.parquet"
CHANNEL_FILE.unlink(missing_ok=True)
channels.to_parquet(CHANNEL_FILE, compression="gzip")

# Video Metadata

In [5]:
@contextmanager
def load_video_meta_archive(channel):
    path = src.PATH / "data/raw/yt" / channel / "metadata.zip"
    archive = None
    try:
        archive = zipfile.ZipFile(path, "r")
        file_list = archive.infolist()
        yield archive, file_list
    finally:
        if archive is not None:
            archive.close()


def load_video_metadata(channel):
    all_videos = []
    with load_video_meta_archive(channel) as (archive, files):
        for file in files:
            with archive.open(file) as f:
                byte_content = f.read()
                content = byte_content.decode("utf-8")
                content = json.loads(content)
                # assert that file exists
                file_path = src.PATH / "data/raw/yt" / channel / "videos" / f"{content['id']}.m4a"
                try:
                    info = {
                        "video_id": content["id"],
                        "channel": channel,
                        "channel_id": content["channel_id"],
                        "video_title": content["title"],
                        "video_duration": content["duration"],
                        "video_views": content["view_count"],
                        "video_likes": content.get("like_count", None),
                        "video_comments": content["comment_count"],
                        "video_was_live": content["is_live"] | content["was_live"],
                        "video_description": content["description"],
                        "video_datetime_upload": dt.datetime.fromtimestamp(content["timestamp"]),
                    }
                except KeyError:
                    print(channel)
                    print(file)
                    raise
                all_videos.append(info)
                if not info["video_was_live"]:
                    assert file_path.is_file(), file_path

    return all_videos


def create_video_frame(channels):
    videos = []
    for channel in channels:
        channel_videos = load_video_metadata(channel)
        videos.extend(channel_videos)

    return pd.DataFrame(videos)

In [6]:
df = create_video_frame(CHANNELS)
df.head()

,video_id,channel,channel_id,video_title,video_duration,video_views,video_likes,video_comments,video_was_live,video_description,video_datetime_upload
0,xczCF4r_5Vw,@spdde,UCSmbK1WtpYn2sOGLvSSXkKw,Rede von Generalsekretär Lars Klingbeil,204,504,10.0,None,False,Der außerordentliche Bundesparteitag stimmt üb...,2018-01-21 16:56:56
1,mpPZvb0TL6o,@spdde,UCSmbK1WtpYn2sOGLvSSXkKw,Roland Kaiser - Meine Bitte,49,3829,103.0,None,False,"Es ist an der Zeit, Nazis und Rassisten zu wid...",2016-11-16 10:55:25
2,2x0Eu6aNGZU,@spdde,UCSmbK1WtpYn2sOGLvSSXkKw,Katarina Barley: So teuer war der Brexit wirkl...,23,13311,188.0,None,False,Wir wollen die arbeitende Mitte entlasten! Das...,2024-01-29 18:00:05
3,zfRLHbgrWpg,@spdde,UCSmbK1WtpYn2sOGLvSSXkKw,Kanzler Olaf Scholz: Wir machen Politik euretw...,60,98233,401.0,None,False,Alles zum ordentlichen Bundesparteitag findest...,2023-12-09 16:27:37
4,nQIhMkwfDyo,@spdde,UCSmbK1WtpYn2sOGLvSSXkKw,Aus der Vergangenheit lernen heißt Zukunft gew...,3676,1341,15.0,None,False,Die erste Rede der Demokratiereise im Rahmen d...,2009-01-08 13:20:15


In [7]:
VIDEO_FILE = src.PATH / "data/yt_metadata/videos.parquet"
VIDEO_FILE.unlink(missing_ok=True)
df.to_parquet(VIDEO_FILE, compression="gzip")